# Lesson 9: Running a Pretrained Language Model

In the earlier Weird AI lessons, you built many of the core pieces of a language model from scratch. In this lesson, you will use a pretrained model so you can focus on the complete text-generation pipeline.

The pipeline is:

```text
prompt text -> tokenizer -> token IDs -> pretrained model -> generated token IDs -> generated text
```

By the end of this notebook, you should be able to explain how a prompt becomes token IDs, how a pretrained model generates more tokens, and how those generated tokens become readable text again.

## Section 1: Setup

Run this notebook from the project root folder. If imports fail, make sure your virtual environment is active and the project is installed in editable mode:

```bash
python -m pip install -e .
python -m pip install -r requirements.txt
```

This notebook uses Hugging Face `transformers`. If it is not installed yet, install it with:

```bash
python -m pip install transformers
```

In [ ]:
from pathlib import Path
import torch

from weird_ai.pretrained import (
    DEFAULT_MODEL_NAME,
    get_device,
    load_tokenizer,
    load_model,
    tokenize_prompt,
    decode_generated_tokens,
    generate_text,
    generate_with_reasoning,
)

from weird_ai.reasoning import build_reasoning_prompt

## Section 2: Check Your Hardware

A GPU is helpful for larger models, but this lesson should work on CPU with a very small demonstration model. The starter code should choose CUDA when available, Apple Silicon MPS when available, and CPU otherwise.

In [ ]:
# TODO: Run this cell after completing get_device() in pretrained.py.

device = get_device()
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Section 3: Load a Tokenizer

A tokenizer converts human-readable text into token IDs. The model does not directly understand strings such as `Write a parody about networking`. It receives integers.

For this notebook, the default model is intentionally tiny so the first download is fast.

In [ ]:
# TODO: Run this cell after completing load_tokenizer() in pretrained.py.

tokenizer = load_tokenizer(DEFAULT_MODEL_NAME)
print(f"Loaded tokenizer for: {DEFAULT_MODEL_NAME}")
print(f"Pad token: {tokenizer.pad_token!r}")
print(f"EOS token: {tokenizer.eos_token!r}")

## Section 4: Tokenization Round Trip

A round trip means:

```text
text -> token IDs -> text
```

This helps verify that the tokenizer can encode and decode text.

In [ ]:
prompt = "Write a parody chorus about a student debugging Python at midnight."

# TODO: Encode the prompt using tokenizer.encode().
token_ids = None

# TODO: Decode the token IDs back into text using tokenizer.decode().
decoded_text = None

print("Original text:")
print(prompt)
print("
Token IDs:")
print(token_ids)
print("
Decoded text:")
print(decoded_text)

### Reflection Check

Answer in your assignment document:

Why does the model need token IDs instead of regular text?

## Section 5: Inspect Individual Tokens

Many modern tokenizers use subword tokens. A word may be one token, or it may be split into multiple pieces.

In [ ]:
# TODO: After token_ids contains real token IDs, print each token ID and the text it decodes to.

for token_id in token_ids:
    piece = tokenizer.decode([token_id])
    print(f"{token_id:>6} -> {piece!r}")

## Section 6: Load a Pretrained Model

A pretrained model already has learned statistical patterns from a large training corpus. In this lesson, you are not training the model. You are using it for inference.

This can take a moment the first time because the model weights need to download.

In [ ]:
# TODO: Run this cell after completing load_model() in pretrained.py.

model = load_model(DEFAULT_MODEL_NAME, device=device)
print(f"Loaded model for: {DEFAULT_MODEL_NAME}")

## Section 7: Generate Text

Now you will complete the full pipeline:

```text
prompt -> tokenizer -> model.generate() -> decode
```

The `generate_text()` function in `pretrained.py` should handle those steps.

In [ ]:
# TODO: Run this cell after completing generate_text() in pretrained.py.

prompt = "Write a short parody chorus about forgetting to commit code before a demo:"

output = generate_text(
    prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(output)

## Section 8: Compare Temperature Settings

Temperature changes how random generation is.

Lower temperature usually produces safer, more predictable text. Higher temperature usually produces more varied, surprising, and sometimes less coherent text.

In [ ]:
prompt = "Write one funny line about a Wi-Fi router with stage fright:"

temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    print("=" * 70)
    print(f"Temperature: {temp}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=temp,
        top_k=50,
        device=device,
    ))
    print()

## Section 9: Compare Top-k Settings

Top-k sampling limits the model to choosing from only the `k` most likely next tokens.

Smaller values restrict the model more. Larger values allow more variety.

In [ ]:
prompt = "Write one strange parody lyric about a database that sings backup vocals:"

top_k_values = [10, 50, 100]

for k in top_k_values:
    print("=" * 70)
    print(f"Top-k: {k}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=0.8,
        top_k=k,
        device=device,
    ))
    print()

## Section 10: Generate with a Reasoning Prompt

In Lesson 8, you built prompt helpers that ask Weird AI to think step-by-step. Now you can combine that reasoning prompt with a pretrained model.

In [ ]:
user_prompt = "Create a parody song idea about students learning recursion."

reasoning_prompt = build_reasoning_prompt(user_prompt)
print(reasoning_prompt)

In [ ]:
# TODO: Run this cell after completing generate_with_reasoning() in pretrained.py.

reasoning_output = generate_with_reasoning(
    user_prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(reasoning_output)

## Section 11: Weird AI Experiment

Choose one of your own parody topics. Generate at least two outputs using different settings. Save the prompts, settings, and outputs in your assignment document.

In [ ]:
# TODO: Replace this with your own topic.
my_topic = "a student who trained an AI model overnight and forgot to save the checkpoint"

# TODO: Try at least two combinations of temperature and top_k.
settings = [
    {"temperature": 0.5, "top_k": 25},
    {"temperature": 1.0, "top_k": 100},
]

for setting in settings:
    print("=" * 70)
    print(setting)
    print("=" * 70)
    print(generate_text(
        f"Write a short parody verse about {my_topic}:",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=80,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        device=device,
    ))
    print()

## Section 12: Lesson Summary

In this notebook, you practiced:

- Loading a pretrained tokenizer
- Encoding text into token IDs
- Decoding token IDs back into text
- Loading a pretrained causal language model
- Generating text from a prompt
- Comparing temperature and top-k settings
- Reusing a Lesson 8 reasoning prompt with a pretrained model

Before submitting, run the unit tests:

```bash
python -m pytest tests/test_pretrained.py
```